# Dataset Selection Summary

Aggregates the KEEP/DROP verdicts from notebooks 01-10. Each notebook looks at one dataset from Systematic Review /data/ and decides whether it belongs in the warfarin dose/INR prediction model pipeline (real patients, has genotype and/or clinical history, has an outcome to predict) or is context-only, reference, or simulation material.

Run notebooks 01-10 first (or re-run them) to generate dataset_decisions.json.

In [ ]:
import json
import pandas as pd
from pathlib import Path

decisions = json.loads(Path("dataset_decisions.json").read_text())
summary = pd.DataFrame(decisions).T
summary.index.name = "dataset"
summary

## Datasets selected for the modeling pipeline

In [ ]:
keep = summary[summary["verdict"].str.startswith("KEEP") & ~summary["verdict"].str.contains("reference")]
keep

## Reference-only datasets (kept, but not modeling rows)

In [ ]:
reference = summary[summary["verdict"].str.contains("reference")]
reference

## Dropped datasets

In [ ]:
dropped = summary[summary["verdict"].str.startswith("DROP")]
dropped

## What this means for the next phase

**Modeling pipeline inputs** (real patients, genotype and/or history plus an outcome):
- `cleaned/iwpc_full_6256_clean.csv`: primary cohort. Genotype, clinical, and therapeutic dose (about 6037 rows after dropping missing-target rows).
- `cleaned/iwpc_1780_clean.csv`: secondary cohort. Genotype, clinical, INR, and dose (n=1780, scaled differently, don't pool directly with the 6256 cohort without re-harmonizing).
- `cleaned/mimic3_*_clean.csv`, `cleaned/mimic4_*_clean.csv`, and `cleaned/eicu_*_clean.csv`: real longitudinal INR plus admission/diagnosis history, no genotype (clinical-only validation arm).

**Reference lookups** (consulted, not trained on):
- `pharmgkb_allele_tables/`: genotype to star-allele/diplotype translation.
- `gme_1000genomes_freq/`, `saudi_pgx_ngs/`: population allele-frequency context.

**Excluded from modeling** (kept in the repo, but out of scope for the predictive model):
- `faers_warfarin`: adverse-event reports only, no genotype, dose, or INR fields.
- `nomen_pk`: external simulated PK/PD project.
- `mipd_simulation`: simulator-generated genotype and "optimal dose" ground truth.

Next step: build a joint feature/target table from the KEEP datasets. This means harmonizing IWPC-6256 and IWPC-1780 scaling, and treating MIMIC-III, MIMIC-IV, and eICU as a genotype-free external validation set, before model development.